# Análise: McDonald's Financial Statements (2002-2022)

Este notebook organiza os passos para carregar, limpar, explorar e analisar o conjunto de dados disponível em:
https://www.kaggle.com/datasets/mikhail1681/mcdonalds-financial-statements-2002-2022

## Objetivos
- Baixar e carregar os dados do Kaggle (ou usar arquivo local).
- Limpeza e tratamento de tipos (datas, numéricos).
- Análise exploratória (tendências ao longo do tempo, missing, correlações).
- Cálculo de indicadores financeiros (margens, crescimento, retornos).
- Visualizações interativas e exportação dos resultados limpos.

In [ ]:
# Instala dependências (descomente se necessário)
# !pip install -q kaggle pandas numpy matplotlib seaborn plotly openpyxl

In [ ]:
# Imports básicos para a análise
import os
import glob
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set(style="whitegrid")
pd.set_option('display.max_columns', 200)

## Aquisição dos dados
Instruções: faça download manual do Kaggle ou use a API `kaggle` colocando `kaggle.json` em `~/.kaggle/`. Exemplo de comando para baixar (no terminal):

kaggle datasets download -d mikhail1681/mcdonalds-financial-statements-2002-2022

In [ ]:
# Tenta localizar CSV/ZIP automaticamente na pasta do repositório
root = '.'
files = glob.glob(os.path.join(root, '**', '*.csv'), recursive=True) + glob.glob(os.path.join(root, '**', '*.zip'), recursive=True)
files = [f for f in files if 'kaggle' not in f.lower()]
files[:20]

In [ ]:
# Se encontrar ZIP com os dados, descompacta; se encontrar CSV, carrega o primeiro que fizer sentido
data_path = None
for f in files:
    if f.lower().endswith('.zip'):
        with zipfile.ZipFile(f, 'r') as z:
            z.extractall('data')
        data_path = 'data'
        break
    if f.lower().endswith('.csv'):
        data_path = f
        break

data_path

In [ ]:
# Carrega o(s) CSV(s) encontrados na pasta `data` ou o arquivo direto
candidates = []
if os.path.isdir('data'):
    candidates = glob.glob('data/**/*.csv', recursive=True)
elif data_path and data_path.lower().endswith('.csv'):
    candidates = [data_path]

if not candidates:
    print('Nenhum CSV encontrado automaticamente. Faça o download dos dados do Kaggle e coloque-os na pasta do repositório ou em ./data')
else:
    print('Arquivos CSV encontrados:', candidates)
    # Carrega o primeiro CSV por padrão
    df = pd.read_csv(candidates[0])
    print('Tamanho:', df.shape)
    display(df.head())

## Limpeza e preparação dos dados
Passos sugeridos: identificar colunas de data, converter colunas numéricas (remover `,` e `$`), tratar `NaN`, e normalizar nomes de colunas.

In [ ]:
# Exemplo genérico de limpeza
def clean_numeric(col):
    return pd.to_numeric(col.astype(str).str.replace('\,', '').str.replace('\$', '').str.replace('(', '-').str.replace(')', ''), errors='coerce')

if 'df' in globals():
    # mostra info inicial
    display(df.info())
    # tenta detectar coluna de data
    date_cols = [c for c in df.columns if 'date' in c.lower() or 'year' in c.lower()]
    date_cols
    if date_cols:
        for c in date_cols:
            df[c] = pd.to_datetime(df[c], errors='coerce')
    # tenta converter colunas numéricas comuns
    for c in df.columns:
        if df[c].dtype == object:
            sample = df[c].dropna().astype(str).head(5).tolist()
            if any(('$' in s or ',' in s or '(' in s or ')' in s) for s in sample):
                df[c] = clean_numeric(df[c])
    display(df.head())
else:
    print('Nenhum dataframe carregado para limpar.')

## Análise exploratória (EDA)
- Tendências ao longo do tempo (receita, lucro líquido, ativos etc.).
- Missing data por coluna e por ano.
- Correlações entre principais indicadores.

In [ ]:
if 'df' in globals():
    # Exemplo: agrupar por ano se existir coluna de data
    if any(np.issubdtype(dt, np.datetime64) for dt in df.dtypes):
        # tenta identificar a primeira coluna datetime
        date_col = [c for c in df.columns if np.issubdtype(df[c].dtype, np.datetime64)][0]
        df['year'] = df[date_col].dt.year
        yearly = df.groupby('year').sum(numeric_only=True).reset_index()
        display(yearly.head())
        # plot exemplo
        num_cols = yearly.select_dtypes(include=[np.number]).columns.tolist()[:4]
        for col in num_cols:
            plt.figure(figsize=(10,4))
            sns.lineplot(data=yearly, x='year', y=col)
            plt.title(f'Trend: {col}')
            plt.show()
    else:
        print('Nenhuma coluna de data detectada para análise por ano.')
else:
    print('Carregue o dataframe antes de executar a EDA.')

## Indicadores financeiros sugeridos
- Margem bruta, margem líquida, crescimento ano-a-ano, crescimento composto (CAGR).
- Retornos sobre ativos e patrimônio (se existirem as colunas correspondentes).
- Análise de volatilidade e sazonalidade.

In [ ]:
# Exemplo de cálculo de margens se existirem colunas chamadas 'Revenue' e 'Net Income'
if 'df' in globals():
    for col in ['Revenue', 'Net Income', 'Total Assets', 'Total Equity']:
        if col not in df.columns:
            print(f'Coluna {col} não encontrada; ajuste os nomes conforme o dataset.')
    if 'Revenue' in df.columns and 'Net Income' in df.columns:
        df['net_margin'] = df['Net Income'] / df['Revenue']
        display(df[['Revenue','Net Income','net_margin']].head())
else:
    print('Carregue e normalize o dataframe antes de calcular indicadores.')

## Salvando resultados
- Salve o conjunto limpo em `data/cleaned_mcdo.csv` e exporte gráficos conforme necessário.

In [ ]:
if 'df' in globals():
    os.makedirs('data', exist_ok=True)
    df.to_csv('data/cleaned_mcdo.csv', index=False)
    print('Dados limpos salvos em data/cleaned_mcdo.csv')
else:
    print('Nada para salvar. Carregue e limpe o dataframe primeiro.')

---
Próximos passos sugeridos:
- Ajustar nomes de colunas para os nomes reais do CSV baixado.
- Implementar cálculos detalhados de indicadores e análises por segmento (se houver).
- Criar dashboards interativos com Plotly Dash ou Streamlit para apresentação.